In [12]:
import pandas as pd

In [13]:
land = pd.read_csv(
    "../raw/statewise land cost.csv"
)
dim_state = pd.read_csv("../clean/dim_state.csv")
print(land.shape)
print(land.columns.tolist())
print(land.head(15).to_string(index=False))
print("\nData types:")
print(land.dtypes)
print("\nMissing values:")
print(land.isna().sum())

(12, 5)
['State', 'Lowest (Rs/Acre)', 'Average (Rs/Acre)', 'Highest (Rs/Acre)', '5-Year CAGR']
         State Lowest (Rs/Acre) Average (Rs/Acre) Highest (Rs/Acre) 5-Year CAGR
     Rajasthan          ₹50,000         ₹8,00,000        ₹30,00,000         14%
Madhya Pradesh        ₹2,00,000         ₹5,00,000        ₹25,00,000         18%
 Uttar Pradesh        ₹3,00,000        ₹12,00,000        ₹80,00,000         15%
       Gujarat        ₹4,00,000        ₹15,00,000        ₹80,00,000         10%
         Bihar        ₹4,00,000        ₹10,00,000        ₹40,00,000         12%
   Maharashtra        ₹5,00,000        ₹18,00,000      ₹1,50,00,000         12%
     Karnataka        ₹6,00,000        ₹20,00,000      ₹1,50,00,000         11%
    Tamil Nadu        ₹8,00,000        ₹25,00,000      ₹1,00,00,000          9%
     Telangana        ₹5,00,000        ₹15,00,000      ₹1,00,00,000         16%
        Punjab       ₹15,00,000        ₹40,00,000      ₹1,00,00,000          8%
       Haryana       ₹10,

In [14]:
cost_cols = [
    "Lowest (Rs/Acre)",
    "Average (Rs/Acre)",
    "Highest (Rs/Acre)"
]

for col in cost_cols:
    land[col] = (
        land[col]
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .astype(float)
    )

land["5-Year CAGR"] = (
    land["5-Year CAGR"]
    .str.replace("%", "", regex=False)
    .str.strip()
    .astype(float)
)

print(land)
print("\nData types:")
print(land.dtypes)

             State  Lowest (Rs/Acre)  Average (Rs/Acre)  Highest (Rs/Acre)  \
0        Rajasthan           50000.0           800000.0          3000000.0   
1   Madhya Pradesh          200000.0           500000.0          2500000.0   
2    Uttar Pradesh          300000.0          1200000.0          8000000.0   
3          Gujarat          400000.0          1500000.0          8000000.0   
4            Bihar          400000.0          1000000.0          4000000.0   
5      Maharashtra          500000.0          1800000.0         15000000.0   
6        Karnataka          600000.0          2000000.0         15000000.0   
7       Tamil Nadu          800000.0          2500000.0         10000000.0   
8        Telangana          500000.0          1500000.0         10000000.0   
9           Punjab         1500000.0          4000000.0         10000000.0   
10         Haryana         1000000.0          3500000.0         20000000.0   
11          Kerala         1500000.0          5000000.0         

In [16]:
import numpy as np

In [17]:
land = land.rename(columns={
    "State": "state_name",
    "Lowest (Rs/Acre)": "land_cost_low_rs_acre",
    "Average (Rs/Acre)": "land_cost_avg_rs_acre",
    "Highest (Rs/Acre)": "land_cost_high_rs_acre",
    "5-Year CAGR": "land_cost_cagr_pct"
})

# Attach canonical state_id
land = land.merge(
    dim_state[["state_id", "state_name"]],
    on="state_name",
    how="left",
    validate="one_to_one"
)

# Derived features
land["land_cost_log_avg"] = np.log1p(
    land["land_cost_avg_rs_acre"]
)

land["land_cost_high_low_spread"] = (
    land["land_cost_high_rs_acre"] -
    land["land_cost_low_rs_acre"]
)

print(land.to_string(index=False))

    state_name  land_cost_low_rs_acre  land_cost_avg_rs_acre  land_cost_high_rs_acre  land_cost_cagr_pct state_id_x state_id_y  land_cost_log_avg  land_cost_high_low_spread
     Rajasthan                50000.0               800000.0               3000000.0                14.0      IN-RJ      IN-RJ          13.592368                  2950000.0
Madhya Pradesh               200000.0               500000.0               2500000.0                18.0      IN-MP      IN-MP          13.122365                  2300000.0
 Uttar Pradesh               300000.0              1200000.0               8000000.0                15.0      IN-UP      IN-UP          13.997833                  7700000.0
       Gujarat               400000.0              1500000.0               8000000.0                10.0      IN-GJ      IN-GJ          14.220976                  7600000.0
         Bihar               400000.0              1000000.0               4000000.0                12.0      IN-BR      IN-BR         

In [18]:
land = land.drop(columns=["state_id_y"])

land = land.rename(columns={
    "state_id_x": "state_id"
})

# Put identifiers first
land = land[
    [
        "state_id",
        "state_name",
        "land_cost_low_rs_acre",
        "land_cost_avg_rs_acre",
        "land_cost_high_rs_acre",
        "land_cost_cagr_pct",
        "land_cost_log_avg",
        "land_cost_high_low_spread"
    ]
]

print(land.to_string(index=False))

state_id     state_name  land_cost_low_rs_acre  land_cost_avg_rs_acre  land_cost_high_rs_acre  land_cost_cagr_pct  land_cost_log_avg  land_cost_high_low_spread
   IN-RJ      Rajasthan                50000.0               800000.0               3000000.0                14.0          13.592368                  2950000.0
   IN-MP Madhya Pradesh               200000.0               500000.0               2500000.0                18.0          13.122365                  2300000.0
   IN-UP  Uttar Pradesh               300000.0              1200000.0               8000000.0                15.0          13.997833                  7700000.0
   IN-GJ        Gujarat               400000.0              1500000.0               8000000.0                10.0          14.220976                  7600000.0
   IN-BR          Bihar               400000.0              1000000.0               4000000.0                12.0          13.815512                  3600000.0
   IN-MH    Maharashtra               50